# 3D Gaussian Splatting from Scratch Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: A 2D Gaussian

We first build a 2D rasteriser. The 3D case reduces to it after projection.

In [ ]:
```python

import torch

import torch.nn as nn

import torch.nn.functional as F

def eval_2d_gaussian(means, covs, points):

    """

    means:  (G, 2)      centres

    covs:   (G, 2, 2)   covariance matrices

    points: (H, W, 2)   pixel coordinates

    returns: (G, H, W)  density at every pixel for every Gaussian

    """

    G = means.size(0)

    H, W, _ = points.shape

    flat = points.view(-1, 2)

    inv = torch.linalg.inv(covs)

    diff = flat[None, :, :] - means[:, None, :]

    d = torch.einsum("gpi,gij,gpj->gp", diff, inv, diff)

    density = torch.exp(-0.5 * d)

    return density.view(G, H, W)

In [ ]:
```

`einsum` does the quadratic form `diff^T Sigma^-1 diff` for every (Gaussian, pixel) pair.

### Step 2: 2D splatting rasteriser

Alpha-compositing front-to-back. Depth in 2D is meaningless, so we use a learned per-Gaussian scalar for order.

In [ ]:
```python

def rasterise_2d(means, covs, colours, opacities, depths, image_size):

    """

    means:     (G, 2)

    covs:      (G, 2, 2)

    colours:   (G, 3)

    opacities: (G,)     in [0, 1]

    depths:    (G,)     per-Gaussian scalar used for ordering

    image_size: (H, W)

    returns:   (H, W, 3) rendered image

    """

    H, W = image_size

    yy, xx = torch.meshgrid(

        torch.arange(H, dtype=torch.float32, device=means.device),

        torch.arange(W, dtype=torch.float32, device=means.device),

        indexing="ij",

    )

    points = torch.stack([xx, yy], dim=-1)

    densities = eval_2d_gaussian(means, covs, points)

    alphas = opacities[:, None, None] * densities

    alphas = alphas.clamp(0.0, 0.99)

    order = torch.argsort(depths)

    alphas = alphas[order]

    colours_sorted = colours[order]

    T = torch.ones(H, W, device=means.device)

    out = torch.zeros(H, W, 3, device=means.device)

    for i in range(means.size(0)):

        a = alphas[i]

        out += (T * a)[..., None] * colours_sorted[i][None, None, :]

        T = T * (1.0 - a)

    return out

In [ ]:
```

Not fast — a real implementation uses tile-based CUDA kernels — but exactly the right math and fully differentiable.

### Step 3: A trainable 2D splat scene

In [ ]:
```python

class Splats2D(nn.Module):

    def __init__(self, num_splats=128, image_size=64, seed=0):

        super().__init__()

        g = torch.Generator().manual_seed(seed)

        H, W = image_size, image_size

        self.means = nn.Parameter(torch.rand(num_splats, 2, generator=g) * torch.tensor([W, H]))

        self.log_scale = nn.Parameter(torch.ones(num_splats, 2) * math.log(2.0))

        self.rot = nn.Parameter(torch.zeros(num_splats))  # single angle in 2D

        self.colour_logits = nn.Parameter(torch.randn(num_splats, 3, generator=g) * 0.5)

        self.opacity_logit = nn.Parameter(torch.zeros(num_splats))

        self.depth = nn.Parameter(torch.rand(num_splats, generator=g))

    def covs(self):

        s = torch.exp(self.log_scale)

        c, si = torch.cos(self.rot), torch.sin(self.rot)

        R = torch.stack([

            torch.stack([c, -si], dim=-1),

            torch.stack([si, c], dim=-1),

        ], dim=-2)

        S = torch.diag_embed(s ** 2)

        return R @ S @ R.transpose(-1, -2)

    def forward(self, image_size):

        covs = self.covs()

        colours = torch.sigmoid(self.colour_logits)

        opacities = torch.sigmoid(self.opacity_logit)

        return rasterise_2d(self.means, covs, colours, opacities, self.depth, image_size)

In [ ]:
```

`log_scale`, `opacity_logit`, and `colour_logits` are all unconstrained parameters mapped through the right activation at render time. This is the standard pattern for every 3DGS implementation.

### Step 4: Fit 2D Gaussians to a target image

In [ ]:
```python

import math

import numpy as np

def make_target(size=64):

    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")

    img = np.zeros((size, size, 3), dtype=np.float32)

    # Red circle

    mask = (xx - 20) ** 2 + (yy - 20) ** 2 < 10 ** 2

    img[mask] = [1.0, 0.2, 0.2]

    # Blue square

    mask = (np.abs(xx - 45) < 8) & (np.abs(yy - 40) < 8)

    img[mask] = [0.2, 0.3, 1.0]

    return torch.from_numpy(img)

target = make_target(64)

model = Splats2D(num_splats=64, image_size=64)

opt = torch.optim.Adam(model.parameters(), lr=0.05)

for step in range(200):

    pred = model((64, 64))

    loss = F.mse_loss(pred, target)

    opt.zero_grad(); loss.backward(); opt.step()

    if step % 40 == 0:

        print(f"step {step:3d}  mse {loss.item():.4f}")

In [ ]:
```

Over 200 steps the 64 Gaussians settle into the two shapes. That is the entire idea — gradient-descent on explicit geometric primitives.

### Step 5: From 2D to 3D

The 3D extension keeps the same loop. The additions:

1. Per-Gaussian rotation is a quaternion instead of a single angle.

2. Covariance is `R S S^T R^T` with `R` built from the quaternion and `S = diag(exp(log_scale))`.

3. Projection `(mu, Sigma) -> (mu', Sigma')` uses the camera extrinsics and the Jacobian of the perspective projection at `mu`.

4. Colour becomes a spherical-harmonics expansion; evaluate it at the viewing direction.

5. Depth-sort is from actual camera-space z instead of a learned scalar.

Every production implementation (`gsplat`, `inria/gaussian-splatting`, `nerfstudio`) does exactly this on the GPU with tile-based CUDA kernels.

### Step 6: Spherical harmonics evaluation

The SH basis up to degree 3 has 16 terms per channel. Evaluation:

In [ ]:
```python

def eval_sh_degree_3(sh_coeffs, dirs):

    """

    sh_coeffs: (..., 16, 3)   last dim is RGB channels

    dirs:      (..., 3)       unit vectors

    returns:   (..., 3)

    """

    C0 = 0.282094791773878

    C1 = 0.488602511902920

    C2 = [1.092548430592079, 1.092548430592079,

          0.315391565252520, 1.092548430592079,

          0.546274215296039]

    x, y, z = dirs[..., 0], dirs[..., 1], dirs[..., 2]

    x2, y2, z2 = x * x, y * y, z * z

    xy, yz, xz = x * y, y * z, x * z

    result = C0 * sh_coeffs[..., 0, :]

    result = result - C1 * y[..., None] * sh_coeffs[..., 1, :]

    result = result + C1 * z[..., None] * sh_coeffs[..., 2, :]

    result = result - C1 * x[..., None] * sh_coeffs[..., 3, :]

    result = result + C2[0] * xy[..., None] * sh_coeffs[..., 4, :]

    result = result + C2[1] * yz[..., None] * sh_coeffs[..., 5, :]

    result = result + C2[2] * (2.0 * z2 - x2 - y2)[..., None] * sh_coeffs[..., 6, :]

    result = result + C2[3] * xz[..., None] * sh_coeffs[..., 7, :]

    result = result + C2[4] * (x2 - y2)[..., None] * sh_coeffs[..., 8, :]

    # degree 3 terms omitted here for brevity; full 16-coefficient version in the code file

    return result

In [ ]:
```

Learned `sh_coeffs` store the "colour in every direction" for that Gaussian. At render time you evaluate against the current view direction and get a 3-vector RGB.

## Exercises

In [ ]:
1. **(Easy)** Run the 2D splat trainer above on a different synthetic image. Vary `num_splats` in `[16, 64, 256]` and plot MSE vs step for each. Identify the point of diminishing returns.
2. **(Medium)** Extend the 2D rasteriser to support per-Gaussian RGB colours that depend on a scalar "view angle" through a degree-2 harmonic. Train on a pair of target images and verify the model reconstructs both.
3. **(Hard)** Clone `nerfstudio` and train `splatfacto` on a 20-photo capture of any scene you have (desk, plant, face, room). Export to glTF `KHR_gaussian_splatting` and open it in a viewer (Three.js `GaussianSplats3D`, SuperSplat, Babylon.js V9). Report training time, number of Gaussians, and rendered fps.